In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import laspy

def concat_pointcloud(directory, chunk_size=100000, save_npy=False, out_dir="processed", tile_name="all_tiles"):
    """
        Loads multiple .las files, processes them in manageable chunks, and aggregates
        the results into a single NumPy array or saves chunks incrementally.

        Arguments: 
            directory:      Path to the data
            chunk_size:     Number of points processed at a time
            save_npy:       If True, saves .npy files chunk by chunk.
            out_dir:        Output directory for .npy files
            tile_name:      Basename for saved files (used as prefix for chunks if save_npy=True)

        Returns:
            If save_npy is False:
                df:             DataFrame with all points
                points_array:   NumPy array with all points
            If save_npy is True:
                None, None (prints paths to saved chunk files)
    """
    files = [os.path.join(directory, f) for f in os.listdir(directory)
             if f.endswith(".las") or f.endswith(".laz")]
    all_points = []  # Only used if save_npy is False
    header = []
    chunk_counter = 0
    saved_files_info = []

    if save_npy:
        os.makedirs(out_dir, exist_ok=True)

    for file_idx, file in enumerate(tqdm(files, desc="Lade .las-Dateien")):
        try:
            las = laspy.read(file)
            dims = list(las.points.point_format.dimension_names)
            scale = las.header.scale
            offset = las.header.offset
            if not header:
                header = dims
            total_points = len(las.points)
        except Exception as e:
            print(f"\nFehler beim Lesen von {file}: {e}")
            continue # Skip to next file

        start = 0
        while start < total_points:
            end = min(start + chunk_size, total_points)
            chunk_data = []
            valid_chunk = True
            for d in dims:
                try:
                    data_array = np.array(las.points[d][start:end], dtype=float)
                    if d in ["X", "Y", "Z"]:
                        idx = ["X", "Y", "Z"].index(d)
                        data_array = data_array * scale[idx] + offset[idx]
                    if data_array.size != (end - start): # Check if read points match expected chunk size
                        print(f"\nWarnung: Inkonsistente Punktzahl in Chunk {chunk_counter} von Datei {file}. Erwartet: {end-start}, Bekommen: {data_array.size}")
                        # Decide how to handle: skip chunk, fill missing, etc. Here we skip.
                        valid_chunk = False
                        break
                    chunk_data.append(data_array)
                except Exception as e:
                    print(f"\nFehler beim Verarbeiten der Dimension '{d}' in Chunk {chunk_counter} von Datei {file}: {e}")
                    valid_chunk = False
                    break

            if not valid_chunk or len(chunk_data) != len(dims):
                start = end
                continue # Skip this chunk

            chunk = np.vstack(chunk_data).T

            if save_npy:
                # Save chunk directly to .npy
                try:
                    # Create temporary DataFrame for easy column selection
                    chunk_df = pd.DataFrame(chunk, columns=header)
                    feature_cols = ["X", "Y", "Z", "intensity", "return_number", "number_of_returns"]
                    label_col = "classification"
                    # Check if all required columns exist in the current chunk's header
                    if not all(col in header for col in feature_cols + [label_col]):
                         print(f"\nWarnung: Überspringe Chunk {chunk_counter}, da benötigte Spalten fehlen ({feature_cols + [label_col]} vs {header})")
                         start = end
                         continue

                    features = chunk_df[feature_cols].to_numpy()
                    labels = chunk_df[label_col].to_numpy().astype(np.int32)

                    points_filename = os.path.join(out_dir, f"{tile_name}_chunk{chunk_counter}_points.npy")
                    labels_filename = os.path.join(out_dir, f"{tile_name}_chunk{chunk_counter}_labels.npy")
                    np.save(points_filename, features)
                    np.save(labels_filename, labels)
                    saved_files_info.append((points_filename, labels_filename))
                    chunk_counter += 1
                except KeyError as e:
                    print(f"\n❌ Fehlende Spalte in Chunk {chunk_counter} zum Speichern: {e}. Überspringe Chunk.")
                except Exception as e:
                    print(f"\n❌ Fehler beim Speichern von Chunk {chunk_counter}: {e}. Überspringe Chunk.")
            else:
                # Original behavior: append to list for later stacking
                all_points.append(chunk)

            start = end

    if save_npy:
        if not saved_files_info:
             print("\nKeine Chunks wurden gespeichert. Überprüfe die Daten und Spaltennamen.")
             return None, None
        else:
             print(f"\n{len(saved_files_info)} Chunks gespeichert in: {out_dir}")
             # Optionally print all filenames:
             # for p, l in saved_files_info:
             #     print(f"  - {os.path.basename(p)}, {os.path.basename(l)}")
             return None, None # Don't return in-memory arrays
    else:
        # Original behavior if not saving chunk by chunk
        if not all_points:
            print("\nKeine Punktdaten gefunden oder verarbeitet.")
            return pd.DataFrame(columns=header), np.array([]) # Return empty structures

        try:
            points_array = np.vstack(all_points)
            df = pd.DataFrame(points_array, columns=header)
            return df, points_array
        except Exception as e:
             print(f"\nFehler beim Zusammenfügen der Chunks im Speicher: {e}")
             print("Möglicherweise ist der Speicher für das gesamte Dataset zu klein. Versuche save_npy=True.")
             # Decide: return partial data, raise error, or return None
             return None, None # Indicate failure


In [ ]:
# Example usage with chunk saving enabled:
result = concat_pointcloud(
    directory="C:/Users/Nicola/Desktop/zurich_laz",
    chunk_size=500000, # Adjust chunk size based on available RAM
    save_npy=True,
    out_dir="processed_chunks", # Save chunks to a specific directory
    tile_name="schweiz_dataset_chunked" # Prefix for chunk files
)

# result will be (None, None) because save_npy=True
# The individual chunk files are saved in 'processed_chunks'

# If you need to process the chunks later, you would load them:
# import glob
# chunk_files = sorted(glob.glob('processed_chunks/*_points.npy'))
# for chunk_file in chunk_files:
#     points_chunk = np.load(chunk_file)
#     # Process points_chunk


Lade .las-Dateien: 100%|██████████| 25/25 [01:41<00:00,  4.06s/it]


MemoryError: Unable to allocate 24.6 GiB for an array with shape (183309863, 18) and data type float64